<a href="https://colab.research.google.com/github/angelrecalde2024/Power-System-Planning-and-Transmission-Design-2026/blob/main/INGP1118_TransmissionExpansion_LinearDC_OPF_LineLimits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LINEAR DECOUPLED DC OPTIMAL POWER FLOW

The optimal power flow does the following:


*   OBJECTIVE: minimize the total cost of generation
*   CONSTRAINTS: 1) Decoupled DC power balance on each bus, 2) Generator limits, 3) slack angle, 4) line limits
*   Power limits ARE enforced in the transmission lines.



This code performs an optimal power flow considering:


*   Linear cost function for the generators

The Locational Marginal Price LMP at each bus is the incremental cost ($/MW) of supplying one additional MW of load at that the bus, given the network constraints. It is the shadow price of the nodal power balance constraint.
A positive LMP means that supplying 1 more MW at that bus increases the total generation cost. This is the normal case because it reflects the marginal generation cost, the congestion effects, and the marginal losses (in AC models).
A negative LMP meas that increasing the load at that bus reduces the total system cost. This occurs when there is overgeneration (must-run units, renewables), congestion preventing export, minimum generation constraints, network topology traps cheap power. Economically, the system would be willing to pay someone to consume power; this happens in real markets (e.g., high wind penetration regions).


*   If there is no congestion, LMP in all buses is the same and equals the dispatch marginal price (single system marginal cost).
*   If congestion exists, LMP on each bus is the dispatch marginal price plus the congestion component.

A higher LMP means constrained import, while lower LMP means constrained export.
If no line hits its limit, all buses have the same LMP. On the constrary, if a line hits its limit, the buses separate into price zones.

In conclusion:

**LMP tells how much the system would pay, or need to be paid, to serve one more MW at that location.**

In [4]:
# =====================================================
# INSTALL
# =====================================================
!pip -q install pyomo highspy pypower openpyxl pandas numpy

# =====================================================
# IMPORTS
# =====================================================
import numpy as np
import pandas as pd
import pyomo.environ as pyo

from pypower.idx_bus import BUS_I, PD
from pypower.idx_gen import GEN_BUS, PMIN, PMAX
from pypower.idx_brch import F_BUS, T_BUS, BR_X, BR_STATUS
from pypower.makeBdc import makeBdc

# =====================================================
# FILE + SETTINGS
# =====================================================
EXCEL = "/content/Data6busSystemTEP.xlsx"   # change if needed
BASE_MVA = 100
SLACK_BUS = 1

# =====================================================
# BUILD PPC (0-based internal indexing, keep Flow limit)
# =====================================================
def build_ppc():

    gen_df  = pd.read_excel(EXCEL, "generator")
    load_df = pd.read_excel(EXCEL, "load")
    line_df = pd.read_excel(EXCEL, "lines")

    # --- headers check (fail fast if something changes) ---
    required_gen = {"generator bus", "a", "b", "c", "Pmin", "Pmax"}
    required_load = {"load bus", "Pload"}
    required_lines = {"from", "to", "X", "Flow limit"}

    if not required_gen.issubset(set(gen_df.columns)):
        raise ValueError(f"generator sheet headers mismatch. Found: {list(gen_df.columns)}")
    if not required_load.issubset(set(load_df.columns)):
        raise ValueError(f"load sheet headers mismatch. Found: {list(load_df.columns)}")
    if not required_lines.issubset(set(line_df.columns)):
        raise ValueError(f"lines sheet headers mismatch. Found: {list(line_df.columns)}")

    # -----------------------------
    # INTERNAL BUS INDEX (0-based)
    # -----------------------------
    buses = sorted(
        set(gen_df["generator bus"])
        | set(load_df["load bus"])
        | set(line_df["from"])
        | set(line_df["to"])
    )

    ext2int = {b:i for i,b in enumerate(buses)}
    int2ext = {i:b for b,i in ext2int.items()}

    nb = len(buses)

    # -----------------------------
    # BUS MATRIX
    # -----------------------------
    bus = np.zeros((nb, 13))
    for i in range(nb):
        bus[i, BUS_I] = i

    for _, r in load_df.iterrows():
        i = ext2int[r["load bus"]]
        bus[i, PD] += r["Pload"]

    # -----------------------------
    # GENERATORS
    # -----------------------------
    ng = len(gen_df)
    gen = np.zeros((ng, 10))
    cost = np.zeros((ng, 3))

    for k, r in gen_df.iterrows():
        gen[k, GEN_BUS] = ext2int[r["generator bus"]]
        gen[k, PMIN] = r["Pmin"]
        gen[k, PMAX] = r["Pmax"]
        cost[k] = [r["a"], r["b"], r["c"]]

    # -----------------------------
    # BRANCH (store X) + LIMITS
    # -----------------------------
    nl = len(line_df)
    branch = np.zeros((nl, 13))
    f_bus = np.zeros(nl, dtype=int)
    t_bus = np.zeros(nl, dtype=int)
    x = np.zeros(nl, dtype=float)
    fmax = np.zeros(nl, dtype=float)

    for k, r in line_df.iterrows():
        f = ext2int[r["from"]]
        t = ext2int[r["to"]]
        f_bus[k] = f
        t_bus[k] = t
        x[k] = float(r["X"])
        fmax[k] = float(r["Flow limit"])

        branch[k, F_BUS] = f
        branch[k, T_BUS] = t
        branch[k, BR_X] = x[k]
        branch[k, BR_STATUS] = 1  # critical (otherwise the line is OFF)

    return {
        "bus": bus,
        "gen": gen,
        "branch": branch,
        "cost": cost,
        "ext2int": ext2int,
        "int2ext": int2ext,
        "slack": ext2int[SLACK_BUS],
        "nb": nb,
        "ng": ng,
        "nl": nl,
        "f_bus": f_bus,
        "t_bus": t_bus,
        "x": x,
        "fmax": fmax
    }

# =====================================================
# DC MATRICES
# =====================================================
def dc_matrices(ppc):
    Bbus, Bf, Pbusinj, _ = makeBdc(
        BASE_MVA,
        ppc["bus"],
        ppc["branch"]
    )
    return np.array(Bbus.todense())

# =====================================================
# DC-OPF WITH LINE LIMITS + DUALS (LMP)
# =====================================================
def solve_dc_opf_with_limits(ppc):

    Bbus = dc_matrices(ppc)

    bus = ppc["bus"]
    gen = ppc["gen"]
    a, b, c = ppc["cost"].T

    nb = ppc["nb"]
    ng = ppc["ng"]
    nl = ppc["nl"]
    slack = ppc["slack"]

    Pd = bus[:, PD]
    gen_bus = gen[:, GEN_BUS].astype(int)

    Pmin = gen[:, PMIN]
    Pmax = gen[:, PMAX]

    f_bus = ppc["f_bus"]
    t_bus = ppc["t_bus"]
    x = ppc["x"]
    fmax = ppc["fmax"]

    print("TOTAL LOAD =", Pd.sum(), "MW")
    print("TOTAL PMAX =", Pmax.sum(), "MW")

    m = pyo.ConcreteModel()

    m.B = pyo.RangeSet(0, nb-1)
    m.G = pyo.RangeSet(0, ng-1)
    m.L = pyo.RangeSet(0, nl-1)

    # variables
    m.theta = pyo.Var(m.B)
    m.Pg = pyo.Var(m.G)

    # dual suffix for LMP / congestion prices
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

    # slack
    m.theta[slack].fix(0)

    # generator limits
    def lim_rule(mdl, g):
        return (float(Pmin[g]), mdl.Pg[g], float(Pmax[g]))
    m.lim = pyo.Constraint(m.G, rule=lim_rule)

    # nodal balance: Pg - Pd = BASE_MVA * Bbus * theta   (MW on both sides)
    def balance(mdl, i):
        Pg_i = sum(mdl.Pg[g] for g in mdl.G if gen_bus[g] == i)
        network = BASE_MVA * sum(Bbus[i, j] * mdl.theta[j] for j in mdl.B)
        return Pg_i - float(Pd[i]) == network
    m.balance = pyo.Constraint(m.B, rule=balance)

    # line flows expression: F = BASE_MVA*(theta_f - theta_t)/X
    def flow_expr(mdl, l):
        return BASE_MVA * (mdl.theta[int(f_bus[l])] - mdl.theta[int(t_bus[l])]) / float(x[l])
    m.F = pyo.Expression(m.L, rule=flow_expr)

    # line limits: -Fmax <= F <= Fmax  (two constraints to extract two duals)
    def flow_ub(mdl, l):
        return mdl.F[l] <= float(fmax[l])
    def flow_lb(mdl, l):
        return -mdl.F[l] <= float(fmax[l])   # equivalent to F >= -Fmax
    m.flow_ub = pyo.Constraint(m.L, rule=flow_ub)
    m.flow_lb = pyo.Constraint(m.L, rule=flow_lb)

    # objective: same as the baseline (linear cost approx): sum b_g * Pg_g
    m.obj = pyo.Objective(expr=sum(float(b[g]) * m.Pg[g] for g in m.G))

    # SOLVE THE PROBLEM
    solver = pyo.SolverFactory("highs")
    res = solver.solve(m)

    Pg = np.array([pyo.value(m.Pg[g]) for g in m.G], dtype=float)
    theta = np.array([pyo.value(m.theta[i]) for i in m.B], dtype=float)
    F = np.array([pyo.value(m.F[l]) for l in m.L], dtype=float)

    # --- LMP extraction ---
    # Sign convention note:
    # For equality constraints in Pyomo, many solvers return duals with the opposite sign
    # relative to the "price" interpretation. Here, we report:
    #   LMP = - dual(balance)
    dual_balance = np.array([m.dual[m.balance[i]] for i in m.B], dtype=float)
    LMP = -dual_balance  # $/MW (given objective in $/h and balance in MW)

    # --- line congestion duals ---
    # If a line is not binding, duals should be ~0.
    mu_ub = np.array([m.dual[m.flow_ub[l]] for l in m.L], dtype=float)  # shadow price of upper limit
    mu_lb = np.array([m.dual[m.flow_lb[l]] for l in m.L], dtype=float)  # shadow price of lower limit

    return Pg, theta, F, LMP, mu_ub, mu_lb

# =====================================================
# POST-PROCESSING: same outputs + limits + LMP + line duals
# =====================================================
def compute_results_with_limits(ppc, Pg, theta, F, LMP, mu_ub, mu_lb):

    gen = ppc["gen"]
    cost = ppc["cost"]
    int2ext = ppc["int2ext"]

    f_bus = ppc["f_bus"]
    t_bus = ppc["t_bus"]
    fmax = ppc["fmax"]

    a, b, c = cost.T

    # generator table with true quadratic cost
    gen_rows = []
    total_cost = 0.0
    for g, P in enumerate(Pg):
        Cg = float(b[g]*P)
        total_cost += Cg
        bus_ext = int2ext[int(gen[g, GEN_BUS])]
        gen_rows.append([g, bus_ext, P, Cg])

    gen_table = pd.DataFrame(gen_rows, columns=["Generator", "Bus", "Pg (MW)", "Cost b*P ($/h)"])

    # angles
    angle_table = pd.DataFrame({
        "Bus": [int2ext[i] for i in range(len(theta))],
        "Theta (rad)": theta,
        "LMP ($/MW)": LMP
    })

    # line flows + limits + yes/no + line duals
    tol = 1e-6
    line_rows = []
    for l in range(len(F)):
        fb_int = int(f_bus[l]); tb_int = int(t_bus[l])
        fb_ext = int2ext[fb_int]; tb_ext = int2ext[tb_int]
        within = "YES" if abs(F[l]) <= float(fmax[l]) + tol else "NO"
        line_rows.append([l, fb_ext, tb_ext, F[l], float(fmax[l]), within, mu_ub[l], mu_lb[l]])

    line_table = pd.DataFrame(
        line_rows,
        columns=[
            "Line",
            "From Bus",
            "To Bus",
            "Flow (MW)",
            "Limit (MW)",
            "Within Limit?",
            "Dual Upper μ⁺ ($/MW)",
            "Dual Lower μ⁻ ($/MW)"
        ]
    )

    return gen_table, angle_table, line_table, total_cost

# =====================================================
# RUN
# =====================================================
ppc = build_ppc()
Pg, theta, F, LMP, mu_ub, mu_lb = solve_dc_opf_with_limits(ppc)

print("\nDISPATCH:")
for g, val in enumerate(Pg):
    bus_ext = ppc["int2ext"][int(ppc["gen"][g, GEN_BUS])]
    print(f"Gen at bus {bus_ext}: {val:.2f} MW")

print("\nANGLES:")
for i, val in enumerate(theta):
    print(f"Bus {ppc['int2ext'][i]} : {val:.5f}")

gen_table, angle_table, line_table, total_cost = compute_results_with_limits(ppc, Pg, theta, F, LMP, mu_ub, mu_lb)

print("\n===== GENERATOR DISPATCH (with cost reporting) =====")
display(gen_table)

print("\n===== BUS ANGLES + LMP =====")
display(angle_table)

print("\n===== LINE FLOWS + LIMITS + DUALS =====")
display(line_table)

print("\nTOTAL SYSTEM COST (linear cost reported) = %.2f $/h" % total_cost)

TOTAL LOAD = 300.0 MW
TOTAL PMAX = 530.0 MW

DISPATCH:
Gen at bus 1: 111.74 MW
Gen at bus 2: 143.26 MW
Gen at bus 3: 45.00 MW

ANGLES:
Bus 1 : 0.00000
Bus 2 : -0.03754
Bus 3 : -0.08870
Bus 4 : -0.09754
Bus 5 : -0.13261
Bus 6 : -0.13728

===== GENERATOR DISPATCH (with cost reporting) =====


,Generator,Bus,Pg (MW),Cost bP ($/h)
0,0,1,111.742203,1303.919767
1,1,2,143.257797,1480.282816
2,2,3,45.000000,487.485000



===== BUS ANGLES + LMP =====


,Bus,Theta (rad),LMP ($/MW)
0,1,0.000000,-11.669000
1,2,-0.037538,-10.333000
2,3,-0.088701,-10.745147
3,4,-0.097538,13.294577
4,5,-0.132613,11.234635
5,6,-0.137277,10.721741



===== LINE FLOWS + LIMITS + DUALS =====


,Line,From Bus,To Bus,Flow (MW),Limit (MW),Within Limit?,Dual Upper μ⁺ ($/MW),Dual Lower μ⁻ ($/MW)
0,0,1,2,18.768884,100.0,YES,-0.000000,-0.0
1,1,1,4,48.768884,100.0,YES,-0.000000,-0.0
2,2,1,5,44.204435,100.0,YES,-0.000000,-0.0
3,3,2,3,20.465279,60.0,YES,-0.000000,-0.0
4,4,2,4,60.000000,60.0,YES,-4.289351,-0.0
5,5,2,5,31.691845,60.0,YES,-0.000000,-0.0
6,6,2,6,49.869557,60.0,YES,-0.000000,-0.0
7,7,3,5,16.889361,60.0,YES,-0.000000,-0.0
8,8,3,6,48.575917,60.0,YES,-0.000000,-0.0
9,9,4,5,8.768884,60.0,YES,-0.000000,-0.0



TOTAL SYSTEM COST (linear cost reported) = 3271.69 $/h
